### Fine-tuning a Frontier Model

In [1]:
# imports

import os
import re
import json
from dotenv import load_dotenv
from huggingface_hub import login
from openai import OpenAI
from pricer.items  import Item
from pricer.evaluator import evaluate

In [2]:
# environment

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_KEY']
login(hf_token, add_to_git_credential=True)

In [3]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


In [4]:
openai = OpenAI()

In [5]:
# OpenAI recommends fine-tuning with populations of 50-100 examples
# But as our examples are very small, I'm suggesting we go with 100 examples (and 1 epoch)


fine_tune_train = train[:50]
fine_tune_validation = val[:25]

In [6]:
len(fine_tune_train)

50

### Data Preparation Step

In [7]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
        {"role": "assistant", "content": f"${item.price:.2f}"}
    ]

In [8]:
messages_for(fine_tune_train[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single‑piece oil‑rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4" minimum center‑to‑center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation.'},
 {'role': 'assistant', 'content': '$64.30'}]

In [9]:
# Convert the items into a list of json objects - a "jsonl" string
# Each row represents a message in the form:
# {"messages" : [{"role": "system", "content": "You estimate prices...


def make_jsonl(items):
    result = ""
    for item in items:
        messages = messages_for(item)
        messages_str = json.dumps(messages)
        result += '{"messages": ' + messages_str +'}\n'
    return result.strip()

In [10]:
print(make_jsonl(train[:3]))

{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Schlage F59 & 613 Andover Interior Knob (Deadbolt Included)  \nCategory: Home Hardware  \nBrand: Schlage  \nDescription: A single\u2011piece oil\u2011rubbed bronze knob that mounts to a deadbolt for secure, easy interior door use.  \nDetails: Designed for a 4\" minimum center\u2011to\u2011center door prep, it offers a lifetime mechanical and finish warranty and comes ready for quick installation."}, {"role": "assistant", "content": "$64.30"}]}
{"messages": [{"role": "user", "content": "Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Mini Electric Air Duster Fan  \nCategory: Electronics  \nBrand: Kica  \nDescription: Ultra\u2011compact 86,000\u202fRPM electric air duster with 11\u202fm/s wind speed for precise cleaning and inflation.  \nDetails: Powered by a 9.99\u202fWh motor, adjustable in four speed levels, it uses three 

In [11]:
# Convert the items into jsonl and write them to a file

def write_jsonl(items, filename):
    with open(filename, "w") as f:
        jsonl = make_jsonl(items)
        f.write(jsonl)

In [12]:
write_jsonl(fine_tune_train, "jsonl/fine_tune_train.jsonl")

In [13]:
write_jsonl(fine_tune_validation, "jsonl/fine_tune_validation.jsonl")

In [14]:
with open("jsonl/fine_tune_train.jsonl", "rb") as f:
    train_file = openai.files.create(file=f, purpose="fine-tune")

In [15]:
train_file

FileObject(id='file-VpigsFENAjLPe6mh8kZU2u', bytes=27472, created_at=1777628966, filename='fine_tune_train.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [16]:
with open("jsonl/fine_tune_validation.jsonl", "rb") as f:
    validation_file = openai.files.create(file=f, purpose="fine-tune")

In [17]:
validation_file

FileObject(id='file-NYUCoYgvriG9Kzfx66Vvuh', bytes=13880, created_at=1777629007, filename='fine_tune_validation.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

### Fine Tunning Step 

In [18]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=validation_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="pricer"
)

FineTuningJob(id='ftjob-wIp1OsmCKWBvp1htUHlMoqqs', created_at=1777629086, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-cZpeh7MW7MXtrdQ8A3qOvqld', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-VpigsFENAjLPe6mh8kZU2u', validation_file='file-NYUCoYgvriG9Kzfx66Vvuh', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_experiment_id=None)

In [19]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-wIp1OsmCKWBvp1htUHlMoqqs', created_at=1777629086, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-cZpeh7MW7MXtrdQ8A3qOvqld', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-VpigsFENAjLPe6mh8kZU2u', validation_file='file-NYUCoYgvriG9Kzfx66Vvuh', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_execution=None, train_experiment_id=None, eval_ex

In [32]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id

In [33]:
job_id

'ftjob-wIp1OsmCKWBvp1htUHlMoqqs'

In [34]:
openai.fine_tuning.jobs.retrieve(job_id)

FineTuningJob(id='ftjob-wIp1OsmCKWBvp1htUHlMoqqs', created_at=1777629086, error=Error(code=None, message=None, param=None), fine_tuned_model='ft:gpt-4.1-nano-2025-04-14:practice:pricer:Daf9Ji5N', finished_at=1777630355, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-cZpeh7MW7MXtrdQ8A3qOvqld', result_files=['file-FkJ9sVQz9ZPP47enWKZKFq'], seed=42, status='succeeded', trained_tokens=5619, training_file='file-VpigsFENAjLPe6mh8kZU2u', validation_file='file-NYUCoYgvriG9Kzfx66Vvuh', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier=0.1, n_epochs=1))), user_provided_suffix='pricer', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None, internal_peashooter_executio

In [35]:
openai.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10).data

[FineTuningJobEvent(id='ftevent-tskBjWkmwd6AJbGFfvZzVUUm', created_at=1777631090, level='info', message='The job has successfully completed', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-3LAD1BpfbptKoY6aTKT6xyrj', created_at=1777631087, level='info', message='Usage policy evaluations completed, model is now enabled for sampling', object='fine_tuning.job.event', data={}, type='message'),
 FineTuningJobEvent(id='ftevent-PR7Tiw0X5VOg3XwUUDSCLd2k', created_at=1777631087, level='info', message='Moderation checks for snapshot ft:gpt-4.1-nano-2025-04-14:practice:pricer:Daf9Ji5N passed.', object='fine_tuning.job.event', data={'blocked': False, 'results': [{'flagged': False, 'category': 'harassment/threatening', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'sexual/minors', 'enforcement': 'blocking'}, {'flagged': False, 'category': 'propaganda', 'enforcement': 'blocking

### Test Fine tune model Step

In [36]:
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [37]:
fine_tuned_model_name

'ft:gpt-4.1-nano-2025-04-14:practice:pricer:Daf9Ji5N'

In [38]:
# The prompt

def test_messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": message},
    ]

In [39]:
# Try this out

test_messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [40]:
# The inference function


def gpt_4__1_nano_fine_tuned(item):
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=test_messages_for(item),
        max_tokens=7
    )
    return response.choices[0].message.content

In [41]:
print(test[0].price)
print(gpt_4__1_nano_fine_tuned(test[0]))

219.0
$395.00


In [42]:
evaluate(gpt_4__1_nano_fine_tuned, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $21 $5 $45 $20 $59 $221 $4 $7 $320 $553 $6 $7 $21 $14 $10 $75 $23 $4 $104 $32 $35 $249 $28 $102 $223 $105 $3 $265 $48 $2 $11 $1 $59 $21 $129 $40 $37 $28 $96 $153 $50 $13 $81 $166 $9 $28 $11 $42 $51 $22 $103 $452 $33 $60 $9 $27 $38 $76 $7 $24 $12 $76 $86 $546 $46 $20 $168 $50 $2 $17 $50 $93 $4 $129 $22 $42 $2 $2 $34 $21 $9 $95 $75 $2 $10 $95 $158 $20 $13 $34 $27 $5 $27 $1 $102 $3 $6 $270 $384 $192 $93 $10 $24 $64 $127 $14 $386 $17 $79 $33 $321 $69 $97 $14 $50 $22 $22 $22 $199 $37 $226 $48 $53 $77 $65 $13 $231 $3 $5 $9 $12 $49 $17 $94 $4 $21 $0 $13 $16 $116 $60 $127 $15 $66 $44 $0 $455 $136 $16 $3 $314 $24 $4359 $17 $405 $246 $26 $55 $5 $610 $15 $6 $1 $441 $15 $42 $25 $21 $30 $2 $65 $187 $15 $189 $446 $32 $91 $48 $43 $835 $47 $201 $102 $3 $8 $87 $26 $26 $3 $15 $192 $88 $73 $52 $250 $41 $460 $5 $6 